Notebook-style lesson: reading tracebacks like a detective.

### Why this matters
Many bugs do not happen in the line that raises the exception. The real mistake
is often higher up the call stack, and the traceback is your map.

### Task
For each case below, predict the exception before running, then test one case at
a time and explain where the real bug sits in the stack.


In [ ]:


from __future__ import annotations


# --- case 1: divide by zero in a helper ---
def average(numbers: list[float]) -> float:
    return sum(numbers) / len(numbers)


def case1() -> float:
    readings: list[float] = []
    return average(readings)


# --- case 2: malformed CSV row ---
def parse_row(row: str) -> dict[str, str]:
    name, age, city = row.split(",")
    return {"name": name, "age": age, "city": city}


def load(rows: list[str]) -> list[dict[str, str]]:
    return [parse_row(r) for r in rows]


def case2() -> list[dict[str, str]]:
    return load(["ada,36,london", "grace,45", "alan,41,cambridge"])


# --- case 3: missing configuration key ---
def get_setting(config: dict[str, object], key: str) -> object:
    return config[key]


def case3() -> object:
    config = {"host": "localhost", "port": 8080}
    return get_setting(config, "timeout")


# --- case 4: type mismatch upstream ---
def total_price(items: list[dict[str, object]]) -> float:
    return sum(item["price"] * item["qty"] for item in items)  # type: ignore[operator]


def case4() -> float:
    cart = [
        {"price": 9.99, "qty": 2},
        {"price": "19.99", "qty": 1},
    ]
    return total_price(cart)


# --- case 5: retry after a failed network call ---
def fetch(url: str) -> str:
    raise ConnectionError(f"could not reach {url}")


def fetch_with_fallback(url: str) -> str:
    try:
        return fetch(url)
    except ConnectionError:
        return fetch(url.replace("https", "http"))


def case5() -> str:
    return fetch_with_fallback("https://example.invalid/data")


if __name__ == "__main__":
    # Uncomment one case at a time and read the traceback before fixing the bug.
    # print(case1())
    # print(case2())
    # print(case3())
    # print(case4())
    # print(case5())
    pass


ANSWERS = """
case 1
  predicted exception : ZeroDivisionError from dividing by zero
  predicted line      : average() / len(numbers)
  actual              : ZeroDivisionError
  frame at fault      : average(), because the list is empty
  fix                 : guard against empty input or validate before dividing

case 2
  predicted exception : ValueError when unpacking a row with too few values
  predicted line      : name, age, city = row.split(",")
  actual              : ValueError
  frame at fault      : parse_row(), but the real bug is in the malformed input row
  fix                 : validate each row before splitting or handle missing columns

case 3
  predicted exception : KeyError
  actual              : KeyError: 'timeout'
  fix                 : use .get() with a default or check for the key before indexing

case 4
  predicted exception : TypeError
  actual              : can't multiply sequence by non-int of type 'float' or similar
  frame at fault      : total_price() raised it, but the real mistake is inside case4 data
  fix                 : normalize form values before using them in arithmetic

case 5
  predicted exception : ConnectionError
  actual              : the final traceback shows the first failure and the retry failure
  note                : the final exception is the important story for debugging; the earlier one explains the retry path
"""
